# 🚀 jDroid-X-Desk: Centralized Cloud Relay & Remote Access Server
**Google Colab Independent Relay & Remote Desktop Bridge**

This notebook runs as a free, 24/7 central signaling and frame relay server in Google Colab.
- Bridges **Desk 1**, **Desk 2**, **Desk 3**, etc. across firewalls and NAT.
- No third-party subscriptions required.
- Works seamlessly with the portable native `jDroid-X-Desk.exe`.

### Step 1: Install Tunnel & Dependencies

In [ ]:
# Download Cloudflare Tunnel for secure public ingress
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print('Cloudflared tunnel ready!')

### Step 2: Write Central Relay Server Script (`colab_relay_server.py`)

In [ ]:
relay_code = '''# jDroid-X-Desk | Central Colab Cloud Relay
import os, sys, json, time
from http.server import HTTPServer, BaseHTTPRequestHandler

DEVICES = {}
FRAME_BUFFERS = {}
INPUT_BUFFERS = {}

class RelayHandler(BaseHTTPRequestHandler):
    def _cors(self):
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')

    def do_OPTIONS(self):
        self.send_response(200)
        self._cors()
        self.end_headers()

    def do_GET(self):
        if self.path in ['/', '/status']:
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'status': 'ONLINE', 'devices': len(DEVICES), 'time': time.time()}).encode())
            return

        if self.path == '/api/relay/devices':
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'success': True, 'devices': list(DEVICES.values())}).encode())
            return

        if self.path.startswith('/api/relay/pull_frame/'):
            target_id = self.path.split('/')[-1]
            frame = FRAME_BUFFERS.get(target_id, None)
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            if frame:
                self.wfile.write(json.dumps({'success': True, 'frame': frame}).encode())
            else:
                self.wfile.write(json.dumps({'success': False, 'empty': True}).encode())
            return

        if self.path.startswith('/api/relay/pull_input/'):
            target_id = self.path.split('/')[-1]
            inputs = INPUT_BUFFERS.pop(target_id, [])
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'success': True, 'inputs': inputs}).encode())
            return

        self.send_response(404)
        self.end_headers()

    def do_POST(self):
        length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(length).decode('utf-8'))

        if self.path == '/api/relay/register':
            d_id = body.get('id')
            if d_id:
                DEVICES[d_id] = {'id': d_id, 'name': body.get('name', 'Desk'), 'last_seen': time.time()}
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'success': True, 'registered': d_id}).encode())
            return

        if self.path == '/api/relay/push_frame':
            target_id = body.get('targetId')
            frame = body.get('frame')
            if target_id and frame:
                FRAME_BUFFERS[target_id] = frame
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'success': True}).encode())
            return

        if self.path == '/api/relay/push_input':
            target_id = body.get('targetId')
            ev = body.get('event')
            if target_id and ev:
                if target_id not in INPUT_BUFFERS:
                    INPUT_BUFFERS[target_id] = []
                INPUT_BUFFERS[target_id].append(ev)
            self.send_response(200)
            self._cors()
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'success': True}).encode())
            return

        self.send_response(404)
        self.end_headers()

print('Starting jDroid-X Relay on port 8088...')
HTTPServer(('0.0.0.0', 8088), RelayHandler).serve_forever()
'''

with open('colab_relay_server.py', 'w') as f:
    f.write(relay_code)
print('colab_relay_server.py successfully written in Colab environment!')

### Step 3: Run Central Relay & Expose Public URL
Click Play on the cell below. It will output a public URL ending in `.trycloudflare.com`.

In [ ]:
import subprocess, time
server = subprocess.Popen(['python3', 'colab_relay_server.py'])
time.sleep(2)
print('\n======================================================')
print('  jDroid-X Colab Central Relay is LIVE!')
print('  Copy the https://*.trycloudflare.com link below:')
print('======================================================\n')
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8088

### (Optional) Step 4: Run Chrome Remote Desktop Headless Host in Colab
If you want to access the Colab Ubuntu virtual desktop directly via `remotedesktop.google.com/access`:

In [ ]:
#@title Setup Chrome Remote Desktop in Colab
CRD_AUTH_CODE = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}

if CRD_AUTH_CODE:
    !apt update -qq
    !wget -q https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
    !dpkg --install chrome-remote-desktop_current_amd64.deb || apt install -y --fix-broken
    !DEBIAN_FRONTEND=noninteractive apt install -y xfce4 desktop-base xfce4-terminal
    !bash -c 'echo "exec /etc/X11/Xsession /usr/bin/xfce4-session" > /etc/chrome-remote-desktop-session'
    !useradd -m -s /bin/bash user
    !echo 'user:root' | chpasswd
    !adduser user sudo
    !adduser user chrome-remote-desktop
    cmd = f'{CRD_AUTH_CODE} --pin={PIN}'
    !su - user -c '{cmd}'
    !service chrome-remote-desktop start
    print('Chrome Remote Desktop Started! Check remotedesktop.google.com/access')
else:
    print('To run CRD, paste the auth command from remotedesktop.google.com/headless into CRD_AUTH_CODE above.')